# RAGTruth Baseline Reproduction (Colab)

This notebook prepares RAGTruth baseline data, runs local Hugging Face generation (no Docker TGI), and computes baseline case-level Precision/Recall/F1.

In [ ]:
import os
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = Path('/content/AIST-FYP')
if not REPO_DIR.exists():
    raise FileNotFoundError(
        'Repository not found at /content/AIST-FYP. Clone or copy your repo there first.'
    )

ARTIFACTS_DIR = Path('/content/drive/MyDrive/AIST-FYP/outputs/evaluation_artifacts')
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print('Drive artifacts folder:', ARTIFACTS_DIR)

%cd /content/AIST-FYP
print('Repo:', REPO_DIR)

In [ ]:
# Install runtime dependencies for baseline-style evaluation in Colab
!python -m pip install -q --upgrade pip
!python -m pip install -q transformers accelerate sentencepiece huggingface_hub tqdm pandas scikit-learn

In [ ]:
# Optional: authenticate for gated models (e.g., Llama-2)
from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
# Step 1: prepare train/dev/test jsonl under benchmark/RAGTruth/baseline
%cd /content/AIST-FYP/benchmark/RAGTruth/baseline
!python prepare_dataset.py
!ls -lh train.jsonl dev.jsonl test.jsonl

In [ ]:
# Optional Step: train baseline model with tqdm progress bar
%cd /content/AIST-FYP
print('Starting baseline training. You should see tqdm progress from Hugging Face Trainer.')
# Uncomment to run training in Colab (requires enough GPU memory and model access)
# !python scripts/run_ragtruth_baseline.py train --profile single-gpu --model-name baseline

In [ ]:
import json
import re
from datetime import datetime

import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
from transformers import AutoTokenizer, AutoModelForCausalLM

BASELINE_DIR = Path('/content/AIST-FYP/benchmark/RAGTruth/baseline')
RAW_DATASET = BASELINE_DIR / 'test.jsonl'
OUTPUT_FILE = BASELINE_DIR / 'prediction.colab.local.jsonl'

RUN_TS = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
METRICS_FILE = ARTIFACTS_DIR / f'ragtruth_baseline_metrics_{RUN_TS}.json'
PREDICTIONS_SYNC_FILE = ARTIFACTS_DIR / f'ragtruth_baseline_predictions_{RUN_TS}.jsonl'

MODEL_NAME_OR_PATH = 'meta-llama/Llama-2-13b-hf'  # change if needed
MAX_SAMPLES = 200  # set None for full split
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.05
TOP_P = 0.95
TOP_K = 40

In [ ]:
TEMPLATES = {
    'QA': (
        'Below is a question:\n'
        '{question}\n\n'
        'Below are related passages:\n'
        '{reference}\n\n'
        'Below is an answer:\n'
        '{response}\n\n'
        'Your task is to determine whether the summary contains either or both of the following two types of hallucinations:\n'
        '1. conflict: instances where the summary presents direct contraction or opposition to the original news;\n'
        '2. baseless info: instances where the generated summary includes information which is not substantiated by or inferred from the original news. \n'
        'Then, compile the labeled hallucinated spans into a JSON dict, with a key \"hallucination list\" and its value is a list of hallucinated spans. If there exist potential hallucinations, the output should be in the following JSON format: {\"hallucination list\": [hallucination span1, hallucination span2, ...]}. Otherwise, leave the value as a empty list as following: {\"hallucination list\": []}.\n'
        'Output:'
    ),
    'Summary': (
        'Below is the original news:\n'
        '{reference}\n\n'
        'Below is a summary of the news:\n'
        '{response}\n'
        'Your task is to determine whether the summary contains either or both of the following two types of hallucinations:\n'
        '1. conflict: instances where the summary presents direct contraction or opposition to the original news;\n'
        '2. baseless info: instances where the generated summary includes information which is not substantiated by or inferred from the original news. \n'
        'Then, compile the labeled hallucinated spans into a JSON dict, with a key \"hallucination list\" and its value is a list of hallucinated spans. If there exist potential hallucinations, the output should be in the following JSON format: {\"hallucination list\": [hallucination span1, hallucination span2, ...]}. Otherwise, leave the value as a empty list as following: {\"hallucination list\": []}.\n'
        'Output:'
    ),
    'Data2txt': (
        'Below is a structured data in the JSON format:\n'
        '{reference}\n\n'
        'Below is an overview article written in accordance with the structured data:\n'
        '{response}\n\n'
        'Your task is to determine whether the summary contains either or both of the following two types of hallucinations:\n'
        '1. conflict: instances where the summary presents direct contraction or opposition to the original news;\n'
        '2. baseless info: instances where the generated summary includes information which is not substantiated by or inferred from the original news. \n'
        'Then, compile the labeled hallucinated spans into a JSON dict, with a key \"hallucination list\" and its value is a list of hallucinated spans. If there exist potential hallucinations, the output should be in the following JSON format: {\"hallucination list\": [hallucination span1, hallucination span2, ...]}. Otherwise, leave the value as a empty list as following: {\"hallucination list\": []}.\n'
        'Output:'
    ),
}

def build_prompt(sample):
    if sample['task_type'] == 'QA':
        return TEMPLATES['QA'].format(
            question=sample['question'],
            reference=sample['reference'],
            response=sample['response'],
        )
    return TEMPLATES[sample['task_type']].format(
        reference=sample['reference'],
        response=sample['response'],
    )

def parse_prediction(text):
    text = text.strip()
    try:
        pred = json.loads(text)
        if isinstance(pred, dict) and 'hallucination list' in pred:
            return pred
    except Exception:
        pass

    match = re.search(r'\{.*\}', text, flags=re.S)
    if match:
        try:
            pred = json.loads(match.group(0))
            if isinstance(pred, dict) and 'hallucination list' in pred:
                return pred
        except Exception:
            pass

    return {'hallucination list': []}

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_OR_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_OR_PATH,
    torch_dtype='auto',
    device_map='auto',
    trust_remote_code=True,
)
model.eval()

In [ ]:
rows = []
with open(RAW_DATASET, 'r', encoding='utf-8') as f:
    for line in f:
        rows.append(json.loads(line))

if MAX_SAMPLES is not None:
    rows = rows[:MAX_SAMPLES]

print(f'Total samples to evaluate: {len(rows)}')
results = []

for item in tqdm(rows):
    prompt = build_prompt(item)
    inst_prompt = f'[INST] {prompt.strip()} [/INST]'

    inputs = tokenizer(inst_prompt, return_tensors='pt').to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
        pad_token_id=tokenizer.pad_token_id,
    )

    gen_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    pred = parse_prediction(gen_text)

    enriched = dict(item)
    enriched['pred'] = pred
    enriched['raw_pred_text'] = gen_text
    results.append(enriched)

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

print('Saved predictions to:', OUTPUT_FILE)

In [ ]:
df = pd.DataFrame.from_records(results)
df['is_halu'] = df['labels'].apply(lambda x: len(x) > 0)
df['pred_halu'] = df['pred'].apply(lambda x: len(x.get('hallucination list', [])) > 0)

overall_recall = recall_score(df['is_halu'], df['pred_halu'])
overall_precision = precision_score(df['is_halu'], df['pred_halu'])
overall_f1 = f1_score(df['is_halu'], df['pred_halu'])

print(f'Overall Case recall/precision/f1: {overall_recall:.3f}, {overall_precision:.3f}, {overall_f1:.3f}')

metrics_payload = {
    'overall': {
        'recall': float(overall_recall),
        'precision': float(overall_precision),
        'f1': float(overall_f1),
        'num_samples': int(len(df)),
    },
    'per_task': {},
    'model_name_or_path': MODEL_NAME_OR_PATH,
    'max_samples': MAX_SAMPLES,
    'prediction_file': str(OUTPUT_FILE),
}

for task in ['QA', 'Summary', 'Data2txt']:
    temp = df[df['task_type'] == task]
    if len(temp) == 0:
        continue
    task_recall = recall_score(temp['is_halu'], temp['pred_halu'])
    task_precision = precision_score(temp['is_halu'], temp['pred_halu'])
    task_f1 = f1_score(temp['is_halu'], temp['pred_halu'])
    print(f'{task} Case recall/precision/f1: {task_recall:.3f}, {task_precision:.3f}, {task_f1:.3f}')
    metrics_payload['per_task'][task] = {
        'recall': float(task_recall),
        'precision': float(task_precision),
        'f1': float(task_f1),
        'num_samples': int(len(temp)),
    }

with open(METRICS_FILE, 'w', encoding='utf-8') as f:
    json.dump(metrics_payload, f, ensure_ascii=False, indent=2)

import shutil
shutil.copy2(OUTPUT_FILE, PREDICTIONS_SYNC_FILE)

print('Saved metrics JSON to:', METRICS_FILE)
print('Synced predictions JSONL to:', PREDICTIONS_SYNC_FILE)